In [ ]:
import re
from pathlib import Path
from collections import Counter

# Create data directory
data_path = Path("data")
data_path.mkdir(exist_ok=True)

raw_file = data_path / "tweets_raw.txt"

# Create a small synthetic file if one doesn't exist
if not raw_file.exists():
    sample_lines = [
        "RT @user1: Check out https://example.com !!! #NLP #AI 😄",
        "Error 500 at 2026-02-11T10:23:45Z from 192.168.0.1 - see http://errors.example.org?id=500",
        "New blog post → https://myblog.org/post/123 #blogging #python",
        "@user2 lol that was wild 😂😂 http://bit.ly/xyz",
        "Just read an amazing paper on #DeepLearning by @researcher99! https://arxiv.org/abs/2401.12345 🤖",
        "@alice @bob check this out — new #transformer model drops today!! https://huggingface.co/models 🔥",
        "RT @ml_daily: State-of-the-art results on #NLProc benchmark https://paperswithcode.com/sota 📈 #AI #ML",
    ]
    raw_file.write_text("\n".join(sample_lines), encoding="utf-8")
    print("Created synthetic tweets_raw.txt")
else:
    print("tweets_raw.txt already exists — using existing file.")

raw_text = raw_file.read_text(encoding="utf-8")
print("\nFirst 3 lines of raw text:")
for line in raw_text.splitlines()[:3]:
    print(" ", line)

Created synthetic tweets_raw.txt

First 3 lines of raw text:
  RT @user1: Check out https://example.com !!! #NLP #AI 😄
  Error 500 at 2026-02-11T10:23:45Z from 192.168.0.1 - see http://errors.example.org?id=500
  New blog post → https://myblog.org/post/123 #blogging #python


#### 1.2.1 Define Regex Patterns

In [ ]:
# URLs: matches http:// and https:// followed by non-whitespace characters
URL_PATTERN = re.compile(r'https?://\S+')

# @mentions: @ followed by one or more word characters (letters, digits, underscore)
MENTION_PATTERN = re.compile(r'@\w+')

# Hashtags: # followed by one or more word characters
HASHTAG_PATTERN = re.compile(r'#\w+')

# Non-ASCII characters: crude approximation for emojis and special symbols
NON_ASCII_PATTERN = re.compile(r'[^\x00-\x7F]+')

# Collapse multiple consecutive whitespace into a single space
MULTISPACE_PATTERN = re.compile(r'\s+')

print("Regex patterns compiled successfully.")
print(f"  URL_PATTERN      : {URL_PATTERN.pattern}")
print(f"  MENTION_PATTERN  : {MENTION_PATTERN.pattern}")
print(f"  HASHTAG_PATTERN  : {HASHTAG_PATTERN.pattern}")
print(f"  NON_ASCII_PATTERN: {NON_ASCII_PATTERN.pattern}")
print(f"  MULTISPACE_PATTERN: {MULTISPACE_PATTERN.pattern}")

Regex patterns compiled successfully.
  URL_PATTERN      : https?://\S+
  MENTION_PATTERN  : @\w+
  HASHTAG_PATTERN  : #\w+
  NON_ASCII_PATTERN: [^\x00-\x7F]+
  MULTISPACE_PATTERN: \s+


#### 1.2.2 Apply Cleaning Pipeline
Normalization scheme:
- Replace URLs → `<URL>`
- Replace mentions → `<USER>`
- Replace hashtags → `<HASHTAG>`
- Replace non-ASCII / emojis → `<EMOJI>`
- Lowercase all text
- Strip leading/trailing whitespace

In [ ]:
def clean_line(line: str) -> str:
    """Apply full normalization pipeline to a single line of text."""
    line = URL_PATTERN.sub("<URL>", line)          # Replace URLs
    line = MENTION_PATTERN.sub("<USER>", line)     # Replace @mentions
    line = HASHTAG_PATTERN.sub("<HASHTAG>", line)  # Replace #hashtags
    line = NON_ASCII_PATTERN.sub("<EMOJI>", line)  # Replace emojis/non-ASCII
    line = line.lower()                             # Lowercase
    line = MULTISPACE_PATTERN.sub(" ", line)       # Normalize whitespace
    return line.strip()                             # Remove leading/trailing spaces

# Apply to all lines
cleaned_lines = [clean_line(ln) for ln in raw_text.splitlines()]

# Display original vs cleaned side-by-side
print("=" * 70)
for original, cleaned in zip(raw_text.splitlines(), cleaned_lines):
    print(f"ORIG : {original}")
    print(f"CLEAN: {cleaned}")
    print("-" * 70)

ORIG : RT @user1: Check out https://example.com !!! #NLP #AI 😄
CLEAN: rt <user>: check out <url> !!! <hashtag> <hashtag> <emoji>
----------------------------------------------------------------------
ORIG : Error 500 at 2026-02-11T10:23:45Z from 192.168.0.1 - see http://errors.example.org?id=500
CLEAN: error 500 at 2026-02-11t10:23:45z from 192.168.0.1 - see <url>
----------------------------------------------------------------------
ORIG : New blog post → https://myblog.org/post/123 #blogging #python
CLEAN: new blog post <emoji> <url> <hashtag> <hashtag>
----------------------------------------------------------------------
ORIG : @user2 lol that was wild 😂😂 http://bit.ly/xyz
CLEAN: <user> lol that was wild <emoji> <url>
----------------------------------------------------------------------
ORIG : Just read an amazing paper on #DeepLearning by @researcher99! https://arxiv.org/abs/2401.12345 🤖
CLEAN: just read an amazing paper on <hashtag> by <user>! <url> <emoji>
---------------------

#### 1.2.3 Save Cleaned Version

In [ ]:
clean_file = data_path / "tweets_cleaned.txt"
clean_file.write_text("\n".join(cleaned_lines), encoding="utf-8")
print(f"Saved cleaned file: {clean_file}")
print(f"Total lines: {len(cleaned_lines)}")

Saved cleaned file: data/tweets_cleaned.txt
Total lines: 7


#### 1.3.1 Extract from Raw Text

In [ ]:
lines = raw_text.splitlines()

all_urls = []
all_mentions = []
all_hashtags = []

for ln in lines:
    all_urls.extend(URL_PATTERN.findall(ln))
    all_mentions.extend(MENTION_PATTERN.findall(ln))
    all_hashtags.extend(HASHTAG_PATTERN.findall(ln))

print("Extracted URLs:")
for u in all_urls:
    print(f"  {u}")

print("\nExtracted Mentions:")
for m in all_mentions:
    print(f"  {m}")

print("\nExtracted Hashtags:")
for h in all_hashtags:
    print(f"  {h}")

Extracted URLs:
  https://example.com
  http://errors.example.org?id=500
  https://myblog.org/post/123
  http://bit.ly/xyz
  https://arxiv.org/abs/2401.12345
  https://huggingface.co/models
  https://paperswithcode.com/sota

Extracted Mentions:
  @user1
  @user2
  @researcher99
  @alice
  @bob
  @ml_daily

Extracted Hashtags:
  #NLP
  #AI
  #blogging
  #python
  #DeepLearning
  #transformer
  #NLProc
  #AI
  #ML


#### 1.3.2 Unique & Frequency Counts

In [ ]:
print("Top URLs (most common first):")
for url, count in Counter(all_urls).most_common():
    print(f"  [{count}x] {url}")

print("\nTop Mentions (most common first):")
for mention, count in Counter(all_mentions).most_common():
    print(f"  [{count}x] {mention}")

print("\nTop Hashtags (most common first):")
for hashtag, count in Counter(all_hashtags).most_common():
    print(f"  [{count}x] {hashtag}")

Top URLs (most common first):
  [1x] https://example.com
  [1x] http://errors.example.org?id=500
  [1x] https://myblog.org/post/123
  [1x] http://bit.ly/xyz
  [1x] https://arxiv.org/abs/2401.12345
  [1x] https://huggingface.co/models
  [1x] https://paperswithcode.com/sota

Top Mentions (most common first):
  [1x] @user1
  [1x] @user2
  [1x] @researcher99
  [1x] @alice
  [1x] @bob
  [1x] @ml_daily

Top Hashtags (most common first):
  [2x] #AI
  [1x] #NLP
  [1x] #blogging
  [1x] #python
  [1x] #DeepLearning
  [1x] #transformer
  [1x] #NLProc
  [1x] #ML


#### 1.3.3 Optional Extension: Extract IP Addresses and Timestamps from Logs

In [ ]:
# IP address pattern
IP_PATTERN = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')

# ISO 8601 timestamp pattern (e.g., 2026-02-11T10:23:45Z)
TIMESTAMP_PATTERN = re.compile(r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z')

# Date with named capture groups (year, month, day)
DATE_GROUPS_PATTERN = re.compile(r'(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})')

ips = []
timestamps = []

for ln in lines:
    ips.extend(IP_PATTERN.findall(ln))
    timestamps.extend(TIMESTAMP_PATTERN.findall(ln))

print(f"IPs found       : {ips}")
print(f"Timestamps found: {timestamps}")

# Parse structured date fields from timestamps
print("\nStructured date fields from timestamps:")
for ts in timestamps:
    m = DATE_GROUPS_PATTERN.search(ts)
    if m:
        print(f"  Full: {ts} → year={m.group('year')}, month={m.group('month')}, day={m.group('day')}")

IPs found       : ['192.168.0.1']
Timestamps found: ['2026-02-11T10:23:45Z']

Structured date fields from timestamps:
  Full: 2026-02-11T10:23:45Z → year=2026, month=02, day=11


---
## Part 2 – Subword Tokenization with HuggingFace (BPE)

### 2.1 Install and Import

In [ ]:
# Install required libraries (run once)
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

install("tokenizers")
install("transformers")

print("Libraries installed.")

Libraries installed.


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

print("Imports successful.")

Imports successful.


### 2.2 Prepare a Toy Corpus
Reuse cleaned tweets + extra sentences emphasizing rare words and morphology.

In [ ]:
toy_corpus_file = data_path / "toy_corpus.txt"

if not toy_corpus_file.exists():
    extra_sentences = [
        "the water of walden pond is so beautifully clear",
        "low new newer newest lower lowest",
        "subword tokenization helps with rareword rarewords ultra-rareword",
        "renew reset renews renewing renewal",
        "byte pair encoding bpe learns merges for frequent pairs",
        "this corpus is tiny but good enough for demonstration",
        "natural language processing involves tokenization lemmatization and parsing",
        "transformers use attention mechanisms to model long range dependencies",
        "the quick brown fox jumps over the lazy dog",
        "machine learning models require large datasets for pretraining",
        "tokenizers convert text into sequences of integer ids",
        "rare words are often split into subword units by bpe",
        "the vocabulary controls the granularity of the tokenization",
    ]
    corpus_text = "\n".join(cleaned_lines + extra_sentences)
    toy_corpus_file.write_text(corpus_text, encoding="utf-8")
    print(f"Created toy_corpus.txt with {len(cleaned_lines) + len(extra_sentences)} lines.")
else:
    print("toy_corpus.txt already exists — using existing file.")

print("\nFirst 10 lines of corpus:")
for line in toy_corpus_file.read_text(encoding="utf-8").splitlines()[:10]:
    print(f"  {line}")

Created toy_corpus.txt with 20 lines.

First 10 lines of corpus:
  rt <user>: check out <url> !!! <hashtag> <hashtag> <emoji>
  error 500 at 2026-02-11t10:23:45z from 192.168.0.1 - see <url>
  new blog post <emoji> <url> <hashtag> <hashtag>
  <user> lol that was wild <emoji> <url>
  just read an amazing paper on <hashtag> by <user>! <url> <emoji>
  <user> <user> check this out <emoji> new <hashtag> model drops today!! <url> <emoji>
  rt <user>: state-of-the-art results on <hashtag> benchmark <url> <emoji> <hashtag> <hashtag>
  the water of walden pond is so beautifully clear
  low new newer newest lower lowest
  subword tokenization helps with rareword rarewords ultra-rareword


### 2.3 Train a Small BPE Tokenizer (Different Vocabulary Sizes)


#### 2.3.1 Helper Function: Train BPE

In [ ]:
def train_bpe_tokenizer(corpus_files, vocab_size=200, special_tokens=None):
    """
    Train a BPE tokenizer on the given corpus files.

    Args:
        corpus_files : list of Path or str — paths to training text files
        vocab_size   : int — target vocabulary size (including special tokens)
        special_tokens: list of str — tokens to always include in vocabulary

    Returns:
        Tokenizer — a trained HuggingFace BPE tokenizer
    """
    if special_tokens is None:
        special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

    # Initialize BPE model with UNK fallback
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

    # Use whitespace pre-tokenizer (split on spaces before BPE merges)
    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        min_frequency=2,  # Ignore character pairs that appear fewer than 2 times
    )

    tokenizer.train(files=[str(f) for f in corpus_files], trainer=trainer)

    # Add [CLS] / [SEP] post-processor to mimic BERT-style encoding
    tokenizer.post_processor = TemplateProcessing(
        single="[CLS] $A [SEP]",
        pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[
            ("[CLS]", tokenizer.token_to_id("[CLS]")),
            ("[SEP]", tokenizer.token_to_id("[SEP]")),
        ],
    )

    return tokenizer

print("train_bpe_tokenizer() function defined.")

train_bpe_tokenizer() function defined.


#### 2.3.2 Train Two Tokenizers

In [ ]:
# ENLARGE CORPUS — required so BPE can learn enough merges
# to actually differ between vocab_size=200 and vocab_size=2000.
# A corpus of only ~20 lines exhausts all unique character pairs at ~150 tokens,
# making both tokenizers identical. Repeating the text 200× gives BPE
# sufficient frequency statistics to fill the larger vocabulary.
corpus_text = toy_corpus_file.read_text(encoding="utf-8")
# Remove any previous duplications before re-expanding
base_lines = corpus_text.splitlines()[:20]  # keep original 20 lines only
big_text = "\n".join(base_lines * 200)       # ~4000 lines total
toy_corpus_file.write_text(big_text, encoding="utf-8")

print(f"New corpus size: {len(big_text.splitlines())} lines")
print("Corpus enlarged — BPE now has enough data to learn merges up to vocab_size=2000.")

New corpus size: 4000 lines
Corpus enlarged — BPE now has enough data to learn merges up to vocab_size=2000.


In [ ]:
print("Training BPE-Small (vocab_size=200)...")
small_bpe = train_bpe_tokenizer([toy_corpus_file], vocab_size=200)

print("Training BPE-Large (vocab_size=2000)...")
large_bpe = train_bpe_tokenizer([toy_corpus_file], vocab_size=2000)

print(f"\nActual small vocab size : {small_bpe.get_vocab_size()}")
print(f"Actual large vocab size : {large_bpe.get_vocab_size()}")
print("\nNote: Actual size may be smaller than target if corpus is limited.")

Training BPE-Small (vocab_size=200)...
Training BPE-Large (vocab_size=2000)...

Actual small vocab size : 200
Actual large vocab size : 382

Note: Actual size may be smaller than target if corpus is limited.


### 2.4 Compare Tokenization of Rare / Morphologically Rich Words

In [ ]:
test_sentences = [
    "rareword ultrarareword ultra-rareword",
    "renew renews renewing renewal reset",
    "the lower newer lowest newest",
    "hyper-antidisestablishmentarian experiment",
    "multilingual tokenizers sometimes over-segment spanish palabras",
]

def show_encoding(tok, text, label=""):
    """Encode text and display tokens, IDs, and sequence length."""
    encoding = tok.encode(text)
    print(f"  Text  : {text}")
    print(f"  Tokens: {encoding.tokens}")
    print(f"  IDs   : {encoding.ids}")
    print(f"  Length: {len(encoding.tokens)} tokens")
    print()

print("=" * 70)
print("=== SMALL VOCAB BPE (vocab_size=200) ===")
print("=" * 70)
for s in test_sentences:
    show_encoding(small_bpe, s)

print("=" * 70)
print("=== LARGE VOCAB BPE (vocab_size=2000) ===")
print("=" * 70)
for s in test_sentences:
    show_encoding(large_bpe, s)

=== SMALL VOCAB BPE (vocab_size=200) ===
  Text  : rareword ultrarareword ultra-rareword
  Tokens: ['[CLS]', 'rareword', 'ul', 't', 'rar', 'ar', 'eword', 'ul', 'tra', '-', 'rareword', '[SEP]']
  IDs   : [2, 112, 85, 39, 92, 48, 108, 85, 145, 6, 112, 3]
  Length: 12 tokens

  Text  : renew renews renewing renewal reset
  Tokens: ['[CLS]', 'renew', 'renew', 's', 'renew', 'ing', 'renew', 'al', 'res', 'et', '[SEP]']
  IDs   : [2, 97, 97, 38, 97, 69, 97, 99, 129, 121, 3]
  Length: 11 tokens

  Text  : the lower newer lowest newest
  Tokens: ['[CLS]', 'the', 'low', 'er', 'new', 'er', 'low', 'est', 'new', 'est', '[SEP]']
  IDs   : [2, 70, 110, 47, 81, 47, 110, 141, 81, 141, 3]
  Length: 11 tokens

  Text  : hyper-antidisestablishmentarian experiment
  Tokens: ['[CLS]', 'h', 'y', 'p', 'er', '-', 'an', 't', 'i', 'd', 'is', 'est', 'ab', 'l', 'is', 'h', 'm', 'en', 't', 'ar', 'i', 'an', 'e', 'x', 'p', 'er', 'i', 'm', 'en', 't', '[SEP]']
  IDs   : [2, 27, 44, 35, 47, 6, 66, 39, 28, 23, 78, 141, 160

#### Count and Compare Average Token Lengths

In [ ]:
def avg_token_length(tok, sentences):
    """Return (average token count, list of per-sentence token counts)."""
    lengths = [len(tok.encode(s).tokens) for s in sentences]
    return sum(lengths) / len(lengths), lengths

small_avg, small_lens = avg_token_length(small_bpe, test_sentences)
large_avg, large_lens = avg_token_length(large_bpe, test_sentences)

print("Per-sentence token counts:")
print(f"{'Sentence':<55} {'Small':>6} {'Large':>6}")
print("-" * 70)
for sent, s_len, l_len in zip(test_sentences, small_lens, large_lens):
    short = (sent[:52] + "...") if len(sent) > 52 else sent
    print(f"{short:<55} {s_len:>6} {l_len:>6}")
print("-" * 70)
print(f"{'Average':<55} {small_avg:>6.1f} {large_avg:>6.1f}")

Per-sentence token counts:
Sentence                                                 Small  Large
----------------------------------------------------------------------
rareword ultrarareword ultra-rareword                       12     11
renew renews renewing renewal reset                         11      7
the lower newer lowest newest                               11      7
hyper-antidisestablishmentarian experiment                  31     27
multilingual tokenizers sometimes over-segment spani...     39     32
----------------------------------------------------------------------
Average                                                   20.8   16.8


### 2.5 Compare to a Pretrained Tokenizer (GPT-2)

In [ ]:
from transformers import AutoTokenizer

print("Loading GPT-2 tokenizer (pre-trained, not trained by us)...")
gpt_like_tok = AutoTokenizer.from_pretrained("gpt2")
print(f"GPT-2 vocab size: {gpt_like_tok.vocab_size}\n")

print("=" * 70)
print("=== GPT-2 TOKENIZER (pre-trained, 50,257 vocab) ===")
print("=" * 70)
gpt_lens = []
for s in test_sentences:
    toks = gpt_like_tok.tokenize(s)
    print(f"  Text  : {s}")
    print(f"  Tokens: {toks}")
    print(f"  Length: {len(toks)} tokens")
    gpt_lens.append(len(toks))
    print()

gpt_avg = sum(gpt_lens) / len(gpt_lens)
print(f"GPT-2 average token count: {gpt_avg:.1f}")

Loading GPT-2 tokenizer (pre-trained, not trained by us)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT-2 vocab size: 50257

=== GPT-2 TOKENIZER (pre-trained, 50,257 vocab) ===
  Text  : rareword ultrarareword ultra-rareword
  Tokens: ['ra', 'rew', 'ord', 'Ġultr', 'ar', 'are', 'word', 'Ġultra', '-', 'ra', 'rew', 'ord']
  Length: 12 tokens

  Text  : renew renews renewing renewal reset
  Tokens: ['ren', 'ew', 'Ġrenew', 's', 'Ġrenew', 'ing', 'Ġrenewal', 'Ġreset']
  Length: 8 tokens

  Text  : the lower newer lowest newest
  Tokens: ['the', 'Ġlower', 'Ġnewer', 'Ġlowest', 'Ġnewest']
  Length: 5 tokens

  Text  : hyper-antidisestablishmentarian experiment
  Tokens: ['hyper', '-', 'ant', 'idis', 'establishment', 'arian', 'Ġexperiment']
  Length: 7 tokens

  Text  : multilingual tokenizers sometimes over-segment spanish palabras
  Tokens: ['mult', 'ilingual', 'Ġtoken', 'izers', 'Ġsometimes', 'Ġover', '-', 'se', 'gment', 'Ġsp', 'anish', 'Ġpal', 'ab', 'ras']
  Length: 14 tokens

GPT-2 average token count: 9.2


#### Summary Comparison: BPE-Small vs BPE-Large vs GPT-2

In [ ]:
print("Summary — Average tokens per sentence:")
print(f"  BPE-Small (vocab={small_bpe.get_vocab_size():>4}): {small_avg:.2f} tokens/sentence")
print(f"  BPE-Large (vocab={large_bpe.get_vocab_size():>4}): {large_avg:.2f} tokens/sentence")
print(f"  GPT-2     (vocab=50257): {gpt_avg:.2f} tokens/sentence")

print("\nObservations:")
print("  - BPE-Small produces the MOST tokens (more splits per word)")
print("  - BPE-Large produces FEWER tokens (more whole-word coverage)")
print("  - GPT-2 (trained on massive data) handles even rare words gracefully")

Summary — Average tokens per sentence:
  BPE-Small (vocab= 200): 20.80 tokens/sentence
  BPE-Large (vocab= 382): 16.80 tokens/sentence
  GPT-2     (vocab=50257): 9.20 tokens/sentence

Observations:
  - BPE-Small produces the MOST tokens (more splits per word)
  - BPE-Large produces FEWER tokens (more whole-word coverage)
  - GPT-2 (trained on massive data) handles even rare words gracefully
